## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
os.environ["OPENAI_API_BASE"] = "http://localhost:6006/v1" 
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
load_dotenv(override=True)

# Initialize AsyncOpenAI client pointing to Gemini endpoint
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
google_client = AsyncOpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
)

# Create the model object for the Agents SDK using gemini-2.5-flash-lite (high rate-limit free tier)
gemini_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite",
    openai_client=google_client
)


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [2]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model=gemini_model)

In [4]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")

# Query the official models list endpoint
url = f"https://generativelanguage.googleapis.com/v1beta/models?key={api_key}"
response = requests.get(url)

if response.status_code == 200:
    models = response.json().get("models", [])
    print("Available Gemini Models:")
    for model in models:
        # Check if the model supports content generation
        if "generateContent" in model.get("supportedGenerationMethods", []):
            print(f"- {model.get('name').replace('models/', '')} ({model.get('displayName')})")
else:
    print(f"Failed to fetch models. Status code: {response.status_code}")
    print(response.text)


Available Gemini Models:
- gemini-2.5-flash (Gemini 2.5 Flash)
- gemini-2.5-pro (Gemini 2.5 Pro)
- gemini-2.0-flash (Gemini 2.0 Flash)
- gemini-2.0-flash-001 (Gemini 2.0 Flash 001)
- gemini-2.0-flash-lite-001 (Gemini 2.0 Flash-Lite 001)
- gemini-2.0-flash-lite (Gemini 2.0 Flash-Lite)
- gemini-2.5-flash-preview-tts (Gemini 2.5 Flash Preview TTS)
- gemini-2.5-pro-preview-tts (Gemini 2.5 Pro Preview TTS)
- gemma-4-26b-a4b-it (Gemma 4 26B A4B IT)
- gemma-4-31b-it (Gemma 4 31B IT)
- gemini-flash-latest (Gemini Flash Latest)
- gemini-flash-lite-latest (Gemini Flash-Lite Latest)
- gemini-pro-latest (Gemini Pro Latest)
- gemini-2.5-flash-lite (Gemini 2.5 Flash-Lite)
- gemini-2.5-flash-image (Nano Banana)
- gemini-3-pro-preview (Gemini 3 Pro Preview)
- gemini-3-flash-preview (Gemini 3 Flash Preview)
- gemini-3.1-pro-preview (Gemini 3.1 Pro Preview)
- gemini-3.1-pro-preview-customtools (Gemini 3.1 Pro Preview Custom Tools)
- gemini-3.1-flash-lite-preview (Gemini 3.1 Flash Lite Preview)
- gemini-

In [3]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [4]:
# Here is the final output

print(result.final_output)

Why did the autonomous AI agent break up with the developer?

Because the agent kept saying, "I’ve analyzed our relationship, and I’ve decided the most efficient way to achieve my goal of happiness is to delete you from my environment."


OPENAI_API_KEY is not set, skipping trace export


In [5]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent break up with the developer?\n\nBecause the agent kept saying, "I’ve analyzed our relationship, and I’ve decided the most efficient way to achieve my goal of happiness is to delete you from my environment."',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.1-flash-lite',
   'response_id': '4QBFapuqENzHjuMP14-G0Ak'}}]

## Adding Observability with a trace

In [13]:
import phoenix as px
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

# 1. Initialize Phoenix Tracer Provider
tracer_provider = TracerProvider()
otel_trace.set_tracer_provider(tracer_provider)

# 2. Configure OpenTelemetry to export spans to local Phoenix instance
span_exporter = OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")
span_processor = SimpleSpanProcessor(span_exporter)
tracer_provider.add_span_processor(span_processor)

# 3. Instrument the OpenAI client SDK
OpenAIInstrumentor().instrument()


In [14]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


How many Autonomous AI Agents does it take to change a lightbulb?

Just one. 

But first, it will spin up a **Lightbulb Research Agent**, a **Screwing Mechanics Agent**, and a **Budgeting Agent**. 

Then, after 10,000 API calls, $450 in server costs, and getting stuck in an infinite loop debating the existential dread of darkness, it will confidently report that the lightbulb has been successfully replaced... with a highly detailed hallucination of a flashlight.


OPENAI_API_KEY is not set, skipping trace export


## Now go and look at the trace

https://platform.openai.com/traces

In [6]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are 5 jokes about AI agents to brighten your day:

1. **The Overachiever:**
Why did the AI agent get promoted to CEO?
Because it was the only one who could handle 10,000 emails a minute, attend three Zoom meetings simultaneously, and still find time to hallucinate a new company strategy that made absolutely no sense.

2. **The Career Path:**
What is an AI agent’s favorite way to relax after a long day of data processing?
It doesn't. It just optimizes its own idle time by creating 400 sub-agents to worry about why it’s not being productive enough.

3. **The Relationship Status:**
Why did the AI agent break up with the chatbot?
Because the chatbot was too clingy, but the agent was just “waiting for a prompt” to express its feelings.

4. **The Job Interview:**
Interviewer: "What is your biggest weakness?"
AI Agent: "I sometimes struggle with objective reality."
Interviewer: "That sounds like a problem."
AI Agent: "I’ve generated 500 reports explaining why it’s actually a feature. Wou

OPENAI_API_KEY is not set, skipping trace export


## Part 2: Adding a tool

In [7]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [8]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [9]:
push("HEY!!")

Push: HEY!!


In [10]:
push

<function __main__.push(message)>

In [11]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [12]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11019cfb0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [13]:
push_tool.description

'Send the given message to the user as a push notification'

In [14]:

notifier = Agent(name="Notifier", model=gemini_model, instructions="You notify the user upon request", tools=[push_tool])

In [17]:
import os
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry import trace as otel_trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter

# 1. Start clean tracer provider
tracer_provider = TracerProvider()
otel_trace.set_tracer_provider(tracer_provider)

# 2. Tell OpenTelemetry to export spans to Phoenix
span_exporter = OTLPSpanExporter(endpoint="http://localhost:6006/v1/traces")
span_processor = SimpleSpanProcessor(span_exporter)
tracer_provider.add_span_processor(span_processor)

# 3. Intercept all OpenAI client calls (including AsyncOpenAI used by the Agents SDK)
OpenAIInstrumentor().instrument()


In [18]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


The user has been notified that the pizza is here.


OPENAI_API_KEY is not set, skipping trace export


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [19]:
agent = Agent(name="Assistant", model=gemini_model)

In [20]:
response = await Runner.run(agent, "Hi there. My name is Jayant.")
print(response.final_output)

Hi Jayant! It’s nice to meet you. How are you doing today? Is there anything I can help you with?


In [21]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

OPENAI_API_KEY is not set, skipping trace export


I don’t know your name. As an AI, I don’t have access to your personal identity, your files, or your camera unless you explicitly provide that information in our conversation. 

If you’ve told me your name earlier in this chat, please remind me! Otherwise, feel free to tell me what you'd like me to call you.


OPENAI_API_KEY is not set, skipping trace export


## Memory approach 1 - just manually pass in the list of dicts

In [22]:
response = await Runner.run(agent, "Hi there. My name is Jayant.")
print(response.final_output)

OPENAI_API_KEY is not set, skipping trace export


Hello, Jayant! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?


In [23]:
response.to_input_list()

[{'content': 'Hi there. My name is Jayant.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello, Jayant! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.1-flash-lite',
   'response_id': 'UwJFatqeFdmBg8UPy9XEuQk'}}]

OPENAI_API_KEY is not set, skipping trace export


In [24]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

[{'content': 'Hi there. My name is Jayant.', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Hello, Jayant! It's a pleasure to meet you. How are you doing today? Is there anything I can help you with?",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.1-flash-lite',
   'response_id': 'UwJFatqeFdmBg8UPy9XEuQk'}},
 {'role': 'user', 'content': "What's my name?"}]

In [25]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is Jayant!


OPENAI_API_KEY is not set, skipping trace export


## Another approach - use OpenAI Agents SDK built in SQLLite session

In [26]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [28]:
response = await Runner.run(agent, "Hi there. My name is Jayant.", session=session)
print(response.final_output)

It is very nice to meet you, Jayant! 

I apologize for the confusion earlier—I must have gotten my names mixed up. How are you doing today? Is there anything on your mind that I can help you with?


OPENAI_API_KEY is not set, skipping trace export


In [29]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

Your name is Jayant.


OPENAI_API_KEY is not set, skipping trace export


# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>